# 3.5 Anwendung: Modellierung einer Messbrücke

Bisher haben wir LGS aus Energiebilanzen hergeleitet, zum Beispiel beim
Wärmedurchgang durch eine Mehrschichtwand. Dasselbe Vorgehen funktioniert
für elektrische Schaltungen: Statt Energiebilanzgleichungen nutzen wir die
**Kirchhoffschen Regeln**.

In diesem Kapitel modellieren wir eine **Wheatstone-Brückenschaltung**.
Das Messprinzip: Drei der vier Widerstände sind bekannte
Referenzwiderstände; der vierte, $R_4$, ist der **Messwiderstand**, dessen
Wert sich durch eine physikalische Einwirkung ändert, zum Beispiel durch
mechanische Dehnung bei einem Dehnungsmessstreifen oder durch Temperatur
bei einem NTC-Widerstand. Wenn alle vier Widerstände in einem bestimmten
Verhältnis stehen, ist der Querstrom $I$ durch den Brückenwiderstand $R_B$
exakt null: Die Brücke ist **abgeglichen**. Jede Abweichung von $R_4$ vom
Sollwert erzeugt einen messbaren Querstrom, aus dem sich der aktuelle Wert
von $R_4$ zurückrechnen lässt.

Warum dieser Umweg über eine Brückenschaltung? Weil sie empfindlicher ist
als eine direkte Widerstandsmessung: Schon minimale Änderungen von $R_4$
erzeugen einen deutlich messbaren Strom $I$, solange die anderen drei
Widerstände stabil bleiben.

*Wie groß ist $I$ für einen gegebenen Wert von $R_4$, und bei welchem
$R_4$ ist die Brücke abgeglichen?*

## Lernziele

* [ ] Sie können die Kirchhoffsche **Knotenregel** und **Maschenregel**
  auf eine einfache Schaltung anwenden und ein LGS
  $\mathbf{A} \cdot \vec{x} = \vec{b}$ daraus ablesen.
* [ ] Sie können das LGS als Python-Funktion implementieren, die den
  Querstrom $I$ für ein gegebenes $R_4$ zurückgibt.
* [ ] Sie können den Sonderfall der abgeglichenen Brücke physikalisch
  begründen und numerisch überprüfen.

## Die Schaltung

Die Brückenschaltung besteht aus vier Widerständen und einem
Brückenwiderstand $R_B$. Von oben fließt der Gesamtstrom $I_0$; er teilt
sich in einen linken Zweig ($I_1$, $I_2$) und einen rechten Zweig
($I_3$, $I_4$) auf. Der Querstrom $I$ fließt durch $R_B$.

![Wheatstone-Brückenschaltung mit eingezeichneten Stromrichtungen](pics/wheatstone_bruecke.svg)

Wheatstone‑Brückenschaltung mit den vier Brückenwiderständen $R_1, R_2, R_3,
R_4$ und dem Brückenwiderstand $R_B$. Die Pfeile geben die definierten
Zählpfeilrichtungen der Ströme $I_0, I_1, I_2, I_3, I_4$ und des Querstroms $I$
durch $R_B$ an; sie bilden die Grundlage für die Formulierung der Knoten‑
(K1-K3) und Maschengleichungen (M1-M3). Die Spannungsquelle $U_0$ speist die
Brücke von rechts, mit Pluspol oben. (Quelle: eigene Abbildung; Lizenz [CC BY-SA
4.0](https://creativecommons.org/licenses/by-sa/4.0))

Gegebene Größen:
$R_1 = R_2 = R_3 = 100\,\Omega$, $R_B = 10\,\Omega$, $U_0 = 10\,\text{V}$.
Der Messwiderstand $R_4$ ist variabel.

Der Lösungsvektor enthält alle sechs Ströme:
$\vec{x} = (I_0,\, I_1,\, I_2,\, I_3,\, I_4,\, I)^{\top}$.

## Knotenregel: die ersten drei Gleichungen

Wir beginnen mit der **Knotenregel** (1. Kirchhoffsches Gesetz): An
jedem Knoten ist die Summe der zufließenden Ströme gleich der Summe der
abfließenden. Das folgt aus der Ladungserhaltung: Ladung kann sich an
einem Knoten nicht ansammeln.

Jeder Knoten liefert eine Gleichung. Zufließende Ströme erhalten
ein positives Vorzeichen, abfließende ein negatives:

\begin{align*}
I_0 - I_1 - I_3 &= 0 \tag{K1} \\
I_1 - I_2 - I   &= 0 \tag{K2} \\
I_3 + I - I_4   &= 0 \tag{K3}
\end{align*}

Die gegebenen Messwerte und die Beschreibung der Knotenregel übertragen wir in
Python-Code.

In [ ]:
import numpy as np

# --- Gegebene Größen ---
R1 = 100.   # Ohm
R2 = 100.   # Ohm
R3 = 100.   # Ohm
R4 = 200.   # Ohm  (Testwert: Messwiderstand vom Sollwert abgewichen)
RB = 10.    # Ohm  (Brückenwiderstand)
U0 = 10.    # V

# Unbekannte: x = [I0, I1, I2, I3, I4, I]  (6 Ströme, 6 Gleichungen)

# --- Knotenregel (1. Kirchhoffsches Gesetz) ---
# Knotengleichung: zufließende Ströme (+) = abfließende Ströme (-)
# rechte Seite b = 0 (keine externe Quelle an einem Knoten)
#
# K1  oberer Knoten: I0 fließt zu, I1 und I3 fließen ab
#     I0 - I1 - I3 = 0
#     Koeffizienten [I0, I1, I2, I3, I4,  I]: [+1, -1,  0, -1,  0,  0]
#
# K2  Knoten Mitte links: I1 fließt zu, I2 und I fließen ab
#     I1 - I2 - I = 0
#     Koeffizienten:                           [ 0, +1, -1,  0,  0, -1]
#
# K3  Knoten Mitte rechts: I3 und I fließen zu, I4 fließt ab
#     I3 - I4 + I = 0
#     Koeffizienten:                           [ 0,  0,  0, +1, -1, +1]

A_knoten = np.array([
    [+1., -1.,  0., -1.,  0.,  0.],   # K1
    [ 0., +1., -1.,  0.,  0., -1.],   # K2
    [ 0.,  0.,  0., +1., -1., +1.],   # K3
])
print('Knotengleichungen (erste 3 Zeilen von A):')
print(A_knoten)

## Maschenregel: die restlichen drei Gleichungen

Die **Maschenregel** (2. Kirchhoffsches Gesetz) sagt: Die Summe aller
Spannungsabfälle entlang einer geschlossenen Masche ist gleich der
Quellenspannung. Das folgt aus der Energieerhaltung. Der
Spannungsabfall an einem Widerstand berechnet sich nach dem Ohmschen
Gesetz als $U = R \cdot I$.

Wir durchlaufen jede Masche im Uhrzeigersinn. Durchläuft man einen
Widerstand in Richtung des definierten Stroms, ergibt das einen
positiven Beitrag; entgegen der Stromrichtung einen negativen:

\begin{align}
R_1 I_1 + R_2 I_2                   &= U_0 \tag{M1} \\
R_3 I_3 + R_4 I_4                   &= U_0 \tag{M2} \\
R_1 I_1 - R_3 I_3 - R_B I           &= 0   \tag{M3}
\end{align}

In [ ]:
# --- Maschenregel (2. Kirchhoffsches Gesetz) ---
# Maschengleichung: Summe der Spannungsabfälle = Quellenspannung
# Spannungsabfall: U = R * I (Ohmsches Gesetz)
# Widerstand in Stromrichtung durchlaufen:        +R * I
# Widerstand entgegen der Stromrichtung:          -R * I
#
# M1  linke Masche (Uhrzeigersinn: Quelle -> R1 -> R2):
#     R1*I1 + R2*I2 = U0
#     Koeffizienten [I0,  I1,  I2,  I3,  I4,  I]: [ 0,  R1,  R2,   0,   0,   0]  | b = U0
#
# M2  rechte Masche (Uhrzeigersinn: Quelle -> R3 -> R4):
#     R3*I3 + R4*I4 = U0
#     Koeffizienten:                               [ 0,   0,   0,  R3,  R4,   0]  | b = U0
#
# M3  Quermasche (Uhrzeigersinn: R1 -> RB -> R3, keine Quelle):
#     R1 in Richtung I1:  +R1*I1
#     RB entgegen I:      -RB*I   (wir laufen entgegen der definierten I-Richtung)
#     R3 entgegen I3:     -R3*I3
#     R1*I1 - R3*I3 - RB*I = 0
#     Koeffizienten:                               [ 0,  R1,   0, -R3,   0, -RB]  | b = 0

A = np.array([
    [+1., -1.,  0., -1.,   0.,   0.],   # K1
    [ 0., +1., -1.,  0.,   0.,  -1.],   # K2
    [ 0.,  0.,  0., +1.,  -1.,  +1.],   # K3
    [ 0.,  R1,  R2,  0.,   0.,   0.],   # M1
    [ 0.,  0.,  0.,  R3,   R4,   0.],   # M2
    [ 0.,  R1,  0., -R3,   0.,  -RB],   # M3
])
b = np.array([0., 0., 0., U0, U0, 0.])

# --- Lösbarkeit prüfen und lösen ---
print(f'Determinante: {np.linalg.det(A):.4f}')

x = np.linalg.solve(A, b)
I0, I1, I2, I3, I4, I = x

print(f'Gesamtstrom  I0 = {I0*1000:.2f} mA')
print(f'Querstrom    I  = {I*1000:.4f} mA')
print(f'Probe bestanden: {np.allclose(A @ x, b)}')

Der Querstrom $I$ ist bei $R_4 = 200\,\Omega$ ungleich null. Das
Vorzeichen sagt, in welche Richtung $I$ tatsächlich fließt: positiv
bedeutet Fluss in der definierten Zählpfeilrichtung, negativ entgegen.

Wie viele Gleichungen brauchen wir insgesamt? Für 6 Unbekannte genau
6 unabhängige Gleichungen. Drei Knoten und drei Maschen reichen genau
aus. Eine vierte Maschengleichung, etwa für die äußere Gesamtmasche,
wäre eine Linearkombination der drei obigen und würde den Rang der
Matrix nicht erhöhen.

### Mini-Übung 1

Sehen Sie sich die Koeffizientenmatrix $\mathbf{A}$ an und beantworten
Sie die folgenden Fragen zunächst im Kopf, ohne Code auszuführen.

1. In Zeile 4 (M1) steht $A_{4,2} = R_1 = 100$. Aus welcher
   physikalischen Gleichung stammt dieser Eintrag? Was bedeutet er
   inhaltlich?
2. In Zeile 6 (M3) steht $A_{6,4} = -R_3 = -100$. Warum ist dieser
   Eintrag negativ?
3. Die Einträge $b[0] = b[1] = b[2] = 0$, aber $b[3] = b[4] = U_0$.
   Was bedeutet es, wenn ein Eintrag von $\vec{b}$ null ist?
4. Ändern Sie im Code oben $R_4 = 50\,\Omega$ und führen Sie die
   Zelle aus. Ändert sich das Vorzeichen von $I$? Was sagt das
   physikalisch aus?

In [ ]:
# Hier Ihren Code eingeben

## Als Funktion implementieren

Das Muster aus der Zelle oben verallgemeinern wir zu einer Funktion.
$R_4$ wird als Parameter übergeben; alles andere bleibt gleich:

In [ ]:
def solve_bridge(R4, R1=100., R2=100., R3=100., RB=10., U0=10.):
    """Berechnet den Querstrom I durch den Brückenwiderstand R_B.

    Parameter
    ----------
    R4 : float
        Variabler Messwiderstand in Ohm
    R1, R2, R3 : float
        Feste Brückenwiderstände in Ohm (Default: 100)
    RB : float
        Brückenwiderstand in Ohm (Default: 10)
    U0 : float
        Speisespannung in V (Default: 10)

    Rückgabe
    --------
    I : float
        Querstrom durch R_B in Ampere, oder None bei singulärer Matrix
    """
    # Koeffizientenmatrix: gleiche Struktur wie oben, R4 als Parameter
    A = np.array([
        [+1., -1.,  0., -1.,   0.,   0.],   # K1: I0 - I1 - I3 = 0
        [ 0., +1., -1.,  0.,   0.,  -1.],   # K2: I1 - I2 - I  = 0
        [ 0.,  0.,  0., +1.,  -1.,  +1.],   # K3: I3 - I4 + I  = 0
        [ 0.,  R1,  R2,  0.,   0.,   0.],   # M1: R1*I1 + R2*I2 = U0
        [ 0.,  0.,  0.,  R3,   R4,   0.],   # M2: R3*I3 + R4*I4 = U0
        [ 0.,  R1,  0., -R3,   0.,  -RB],   # M3: R1*I1 - R3*I3 - RB*I = 0
    ])
    b = np.array([0., 0., 0., U0, U0, 0.])

    if np.isclose(np.linalg.det(A), 0.):
        print(f'Warnung: singuläre Matrix bei R4 = {R4} Ohm')
        return None

    # Index 5 im Lösungsvektor ist der Querstrom I
    return np.linalg.solve(A, b)[5]

# Test für R4 = 200 Ohm
R4_test = 200.
I_test  = solve_bridge(R4_test)

print(f'R4 = {R4_test:.0f} Ohm')
print(f'Querstrom        I   = {I_test*1000:.4f} mA')
print(f'Verlustleistung  P_B = {RB * I_test**2 * 1000:.4f} mW')

### Mini-Übung 2

Überprüfen Sie `solve_bridge` für den Sonderfall $R_4 = 100\,\Omega$
(alle vier Brückenwiderstände gleich groß).

1. Was erwarten Sie physikalisch für den Querstrom $I$? Überlegen Sie
   zunächst im Kopf: Was passiert mit den Spannungen in der linken und
   rechten Hälfte der Brücke, wenn $R_1 = R_2 = R_3 = R_4$?
2. Rufen Sie `solve_bridge(100.)` auf. Stimmt das Ergebnis mit Ihrer
   Erwartung überein?
3. Erweitern Sie die Funktion oder schreiben Sie eine neue Zelle, die
   den gesamten Lösungsvektor $\vec{x}$ berechnet. Wie groß ist der
   Gesamtstrom $I_0$ bei $R_4 = 100\,\Omega$? Überprüfen Sie das
   Ergebnis mit einer Probe.

In [ ]:
# Hier Ihren Code eingeben

## Zusammenfassung und Ausblick

Das Vorgehen ist dasselbe wie bei der Wärmeübertragung in Kapitel 3.3:
physikalische Gleichungen aufstellen, alle Unbekannten auf die linke
Seite bringen, $\mathbf{A}$ und $\vec{b}$ ablesen. Neu ist nur, dass
die Gleichungen jetzt aus der Knotenregel und der Maschenregel stammen.

Als Funktion `solve_bridge(R4)` verpackt, lässt sich das System für
beliebige $R_4$-Werte auswerten. Im nächsten Kapitel nutzen wir diese
Funktion für eine vollständige Parameterstudie: Wir berechnen $I(R_4)$
und die Verlustleistung $P_B(R_4)$ für viele Werte und stellen die
Ergebnisse in einem Diagramm dar. Dort sehen wir auch die Nullstelle
bei $R_4^* = 100\,\Omega$ und können sie mit der analytischen
Abgleichbedingung vergleichen.